# E-Commerce Revenue & Customer Intelligence Platform

This notebook documents the core analytical workflow for the public Olist Brazilian e-commerce dataset. The production pipeline is in `src/build_analytics.py`; this notebook focuses on transparent business exploration.

## Business definitions

- **Realized GMV:** item price for delivered orders, excluding freight.
- **Repeat customer:** a `customer_unique_id` with at least two delivered orders.
- **On time:** actual customer delivery date is on or before the estimated date.
- **Important:** GMV is not net Olist revenue or profit because commission and cost data are unavailable.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DB = ROOT / 'data' / 'warehouse' / 'ecommerce_analytics.db'
conn = sqlite3.connect(DB)
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)

## Executive KPIs

In [ ]:
executive_sql = '''
SELECT
  COUNT(DISTINCT order_id) AS total_orders,
  SUM(is_delivered) AS delivered_orders,
  ROUND(SUM(realized_gmv), 2) AS realized_gmv_brl,
  ROUND(SUM(realized_gmv) / SUM(is_delivered), 2) AS average_order_value_brl,
  ROUND(AVG(CASE WHEN order_delivered_customer_date IS NOT NULL THEN is_on_time END) * 100, 2) AS on_time_rate_pct,
  ROUND(AVG(review_score), 2) AS average_review_score
FROM fact_orders
'''
pd.read_sql_query(executive_sql, conn)

## Monthly commercial trend

In [ ]:
monthly = pd.read_sql_query('SELECT * FROM monthly_kpis ORDER BY purchase_month', conn)
monthly['purchase_month'] = pd.to_datetime(monthly['purchase_month'])
monthly.loc[monthly['delivered_orders'] >= 500, ['purchase_month', 'delivered_orders', 'realized_gmv', 'aov']].tail(12)

## RFM customer segments

In [ ]:
rfm = pd.read_sql_query('''
SELECT rfm_segment, customers, orders, gmv, customer_share, gmv_share
FROM rfm_summary ORDER BY gmv DESC
''', conn)
rfm.style.format({'gmv': 'R${:,.0f}', 'customer_share': '{:.1%}', 'gmv_share': '{:.1%}'})

## Delivery experience and reviews

In [ ]:
delivery = pd.read_sql_query('SELECT * FROM delivery_review', conn)
delivery[['delay_bucket', 'orders', 'avg_delivery_days', 'avg_review_score']]

## Recommended actions

1. Build a 30–45 day second-purchase journey for recent one-time buyers.
2. Prioritise seller-state combinations with high late-order volume.
3. Test category bundles while monitoring freight, delivery time, and review score.
4. Validate campaign impact with a holdout group instead of relying only on before/after comparison.